In [1]:
from transformers import T5EncoderModel, T5Tokenizer
from Bio import SeqIO
import torch
import numpy as np
import os

In [ ]:
# Loading the ProtT5 model
model_name = "Rostlab/prot_t5_xl_uniref50"
tokenizer = T5Tokenizer.from_pretrained(model_name, do_lower_case=False)
model = T5EncoderModel.from_pretrained(model_name)
print(tokenizer)

In [3]:
def encode_sequence(sequence):
    encoded = tokenizer(sequence, return_tensors='pt')

    with torch.no_grad():
        embedding = model(**encoded)['last_hidden_state'][:,0:-1] # </s>: 1 是结束标记 
    return embedding

In [4]:
def process_fasta(file_path, output_dir):
    embeddings = {}
    with open(file_path, 'r') as fasta_file:
        for record in SeqIO.parse(fasta_file, 'fasta'):
            sequence = str(record.seq)
            print(sequence, len(sequence))
            sequence = " ".join(sequence)
            
            embedding = encode_sequence(sequence)
            embeddings[record.id] = embedding.cpu().numpy()
            print(f"Processed Sequence ID: {record.id}, Embedding Shape: {embedding.shape}")

    # Save the embedding vector
    for seq_id, emb in embeddings.items():
        np.save(f"{output_dir}/{seq_id}.npy", emb)

In [5]:
output_dir = 'example_data'
os.makedirs(output_dir, exist_ok=True)

# Call the function to process the FASTA file and save the embedding vector
file_path = 'example_data.fasta'
process_fasta(file_path, output_dir)

IGLRLPNMLKF 11
Processed Sequence ID: Positive_0, Embedding Shape: torch.Size([1, 11, 1024])
LRSPKMMHKSGCFGRRLDRIGSLSGLGCNVLRKY 34
Processed Sequence ID: Positive_1, Embedding Shape: torch.Size([1, 34, 1024])
SADYLDVSQ 9
Processed Sequence ID: Positive_2, Embedding Shape: torch.Size([1, 9, 1024])
HPVVQSAEMSFGRPVVVEEEQALNPEELSFSEQAYLSHDAAGFGYPSLISGDISSDGLRTAGFVPSQAVKEALLEKPLWSRFLGS 85
Processed Sequence ID: Positive_3, Embedding Shape: torch.Size([1, 85, 1024])
GLYSSERTEEEVEISHGMHHRE 22
Processed Sequence ID: Positive_4, Embedding Shape: torch.Size([1, 22, 1024])
ELEGERPLGLEQVLESDAEKDDGPYRVEHFRWSNPPKDKRYGGFMTSEKSQTPLVTLFKNAIIKNAHKKGQ 71
Processed Sequence ID: Positive_5, Embedding Shape: torch.Size([1, 71, 1024])


In [7]:
import numpy as np

file_path = 'example_data/Positive_0.npy'
data = np.load(file_path)

print(f"data shape: {data.shape}")
print(f"data: \n {data}")

data shape: (1, 11, 1024)
data: 
 [[[ 0.01653975  0.03709732 -0.25645882 ...  0.2900776  -0.09949325
   -0.08135059]
  [ 0.0830673  -0.05559599 -0.35515454 ...  0.25842765  0.10877655
   -0.06174242]
  [-0.10802767  0.0947364   0.01412915 ...  0.05741686  0.07873915
    0.09664293]
  ...
  [-0.07452841  0.26936567  0.16521178 ... -0.03851208  0.07655206
    0.20453152]
  [ 0.17993031 -0.03715436  0.18089764 ...  0.1837725  -0.01940302
   -0.19307105]
  [ 0.14306822  0.04448887  0.0374499  ...  0.13231076 -0.05647983
   -0.16887061]]]
